# Stage-2b — field-mediated neural system: GPU real-time measurement (Colab)

**Before running:** `Runtime → Change runtime type → Hardware accelerator → GPU` (T4 / L4 / A100 — any is fine; the ceiling depends on which). Then `Runtime → Run all`.

This notebook is self-contained: it writes the three source files to disk, installs CuPy, and runs the measurement harness. It prints the hardware report, CPU↔GPU correctness parity, the N-sweep table with per-stage breakdown, the four headline numbers, the Blackwell (96 GB) estimate, and a **measured** real-time-ceiling plot.

**Paste the full text output back to Claude** to fold the measured numbers into the repo.


## 1. Confirm the GPU runtime

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv || echo 'NO GPU — set Runtime to GPU and re-run'

## 2. Install CuPy
Colab's GPU runtime is CUDA 12.x → `cupy-cuda12x`. (Often already present; this is a no-op then.)

In [ ]:
import subprocess, sys
try:
    import cupy; print('cupy already present:', cupy.__version__)
except Exception:
    print('installing cupy-cuda12x ...')
    subprocess.run([sys.executable,'-m','pip','install','-q','cupy-cuda12x'], check=True)
    import cupy; print('cupy installed:', cupy.__version__)


## 3. Write the source files
Stage-1 (`field_system.py`) and Stage-2a (`field_system_v2.py`) are imported by the Stage-2b harness for the correctness-parity cross-check; Stage-2b (`field_system_v2_gpu.py`) is the CuPy port + measurement.

In [ ]:
%%writefile field_system.py
#!/usr/bin/env python3
"""
Field-Mediated Neural System — Stage 1 CPU Reference Implementation
===================================================================

"Neurons as atoms in a field." Discrete UNITS (fixed positions) never see one
another; they only DEPOSIT into a shared continuous scalar FIELD and READ back
from it. ALL coupling is mediated by the field, which evolves by
diffusion-with-decay driven by the unit sources:

        ∂φ/∂t = D·∇²φ − λ·φ + S            (S = sum of unit sources)

This is a correctness-first reference: NumPy + matplotlib only, no GPU, no
optimization tricks, no learning, no unit motion. The goal is to *verify*
(falsifiably) that a shared diffusion field can carry signal between units,
with the right stability, locality, causality and conservation behaviour.

One tick = four operations, in THIS exact order:

    1. DEPOSIT  S(x) = Σ_i a_i · G(x − x_i)          G: normalized 2D Gaussian
    2. EVOLVE   φ ← φ + Δt·(D·∇²φ − λ·φ + S)         explicit Euler, Neumann BC
    3. READ     φ_i = ∫ G(x − x_i)·φ(x) dA           SAME kernel  ⇒ symmetric
    4. UPDATE   s_i ← s_i + Δt·(−γ·s_i + φ_i);  a_i ← tanh(s_i)

Why read uses the same kernel G (the symmetry that makes this a real field
theory): with deposit  S = G·a  and read  φ = h²·Gᵀ·Φ, the read operator is
exactly h²·(deposit)ᵀ. The steady-state unit→unit coupling is therefore
h²·Gᵀ·M·G with M = (λI − D·L)⁻¹; because the Neumann 5-point Laplacian L is a
symmetric (graph-Laplacian) matrix, M is symmetric and so the coupling i↔j is
reciprocal. Action through the field obeys Newton's third law.

Run:   python field_system.py
Deps:  numpy, matplotlib   (only)
"""

from __future__ import annotations

import time
from dataclasses import dataclass, replace

import numpy as np
import matplotlib
matplotlib.use("Agg")              # headless backend: write PNGs, never open a window
import matplotlib.pyplot as plt


# ----------------------------------------------------------------------------
# Configuration
# ----------------------------------------------------------------------------
@dataclass
class Config:
    """All parameters in one place. Defaults are the Stage-1 spec defaults."""
    H: int = 128                   # field grid height (rows, y)
    W: int = 128                   # field grid width  (cols, x)
    h: float = 1.0                 # grid spacing (physical length per cell)
    sigma: float = 1.5             # Gaussian deposit/read std  σ
    D: float = 0.2                 # diffusion coefficient  D
    lam: float = 0.05              # field decay rate  λ
    gamma: float = 0.1             # unit-state leak rate  γ
    dt: float = 1.0                # time step  Δt
    N: int = 10000                 # number of units
    seed: int = 0                  # RNG seed for unit placement
    kernel_radius_sigmas: float = 4.0   # Gaussian window half-width, in units of σ

    # --- derived quantities -------------------------------------------------
    @property
    def influence_range(self) -> float:
        """Screening / influence length of the field:  L = sqrt(D/λ).

        This is the length scale of the steady screened-diffusion Green's
        function K0(r/L): the field from a point source is large within r≈L
        and exponentially negligible beyond a few L."""
        return float(np.sqrt(self.D / self.lam))

    @property
    def dt_max(self) -> float:
        """Hard explicit-Euler stability bound for diffusion:  Δt ≤ h²/(4D)."""
        return self.h ** 2 / (4.0 * self.D)

    def describe(self) -> str:
        return (
            "  grid           H×W      = {H}×{W}  (h={h})\n"
            "  Gaussian       σ        = {sig}\n"
            "  diffusion      D        = {D}\n"
            "  field decay    λ        = {lam}\n"
            "  unit leak      γ        = {gam}\n"
            "  time step      Δt       = {dt}\n"
            "  units          N        = {N}   (seed={seed})\n"
            "  ----------------------------------------------------\n"
            "  influence range  L = sqrt(D/λ)        = {L:.4f}\n"
            "  stability bound  Δt_max = h²/(4D)     = {dtm:.4f}\n"
        ).format(H=self.H, W=self.W, h=self.h, sig=self.sigma, D=self.D,
                 lam=self.lam, gam=self.gamma, dt=self.dt, N=self.N,
                 seed=self.seed, L=self.influence_range, dtm=self.dt_max)


def assert_stable(cfg: Config) -> None:
    """HARD STABILITY CONSTRAINT. Refuse to run if Δt > h²/(4D)."""
    if cfg.dt > cfg.dt_max + 1e-12:
        print("\n*** REFUSING TO RUN — explicit Euler would be unstable ***")
        print(f"    Δt = {cfg.dt} violates  Δt ≤ h²/(4D) = {cfg.dt_max:.6f}")
        print(f"    Reduce Δt to at most {cfg.dt_max:.6f} and try again.")
        raise SystemExit(1)


# ----------------------------------------------------------------------------
# The field system
# ----------------------------------------------------------------------------
class FieldSystem:
    """Holds the field Φ, the units (fixed positions, state s, emission a) and
    the config. The four operations are deposit / evolve / read / update, and
    step() composes them in order. The ONLY persistent state is (phi, s, a):
    the transient source S and the read-back φ_i are passed explicitly between
    methods, never stashed on the object (so step() has no hidden mutation).
    """

    def __init__(self, cfg: Config, positions: np.ndarray | None = None):
        assert_stable(cfg)                     # safety: never build an unstable system
        self.cfg = cfg

        # --- the field: scalar Φ on an H×W grid, Φ[row, col] = Φ[y, x] -------
        self.phi = np.zeros((cfg.H, cfg.W), dtype=np.float64)

        # --- unit positions (physical coords, x=col, y=row) -----------------
        rng = np.random.default_rng(cfg.seed)
        if positions is None:
            # uniform random in the interior, kept ≥3σ from every edge
            m = 3.0 * cfg.sigma
            xs = rng.uniform(m * cfg.h, (cfg.W - 1) * cfg.h - m * cfg.h, cfg.N)
            ys = rng.uniform(m * cfg.h, (cfg.H - 1) * cfg.h - m * cfg.h, cfg.N)
            positions = np.stack([xs, ys], axis=1)
        self.pos = np.asarray(positions, dtype=np.float64)      # (N, 2): (x, y)
        self.N = self.pos.shape[0]

        # --- unit state -----------------------------------------------------
        self.s = np.zeros(self.N, dtype=np.float64)             # internal state s_i
        self.a = np.zeros(self.N, dtype=np.float64)             # emission a_i
        self.clamped = np.zeros(self.N, dtype=bool)             # clamped ⇒ skip update()

        # --- precompute the per-unit Gaussian windows (positions are fixed) --
        self._build_kernels()

    # ---- kernel precomputation --------------------------------------------
    def _build_kernels(self) -> None:
        """For each unit, precompute a local Gaussian window: the field indices
        it touches and the (normalized) kernel weights. Positions never move, so
        this is done once. The kernel is the normalized 2D Gaussian

            G(r) = (1 / 2πσ²) · exp(−|r|² / 2σ²),     ∫ G dA = 1,

        sampled on the grid and renormalized so the *discrete* integral
        Σ G·h² = 1 exactly. Renormalizing per-unit also makes truncation at the
        window edge and clipping at the grid boundary mass-conserving, and (since
        the diffusion propagator is symmetric) leaves the i↔j coupling reciprocal.
        """
        cfg = self.cfg
        rad = int(np.ceil(cfg.kernel_radius_sigmas * cfg.sigma / cfg.h))   # window half-width (cells)
        ww = 2 * rad + 1
        offs = np.arange(-rad, rad + 1)                         # (ww,) cell offsets

        px = self.pos[:, 0]                                     # (N,) x (col) physical
        py = self.pos[:, 1]                                     # (N,) y (row) physical
        cc = np.round(px / cfg.h).astype(np.int64)             # (N,) nearest center col
        cr = np.round(py / cfg.h).astype(np.int64)             # (N,) nearest center row

        cols = cc[:, None] + offs[None, :]                     # (N, ww) integer col indices
        rows = cr[:, None] + offs[None, :]                     # (N, ww) integer row indices

        # signed distances (physical) from each window cell center to the unit
        dx = cols * cfg.h - px[:, None]                        # (N, ww) along x (cols)
        dy = rows * cfg.h - py[:, None]                        # (N, ww) along y (rows)
        dist2 = dy[:, :, None] ** 2 + dx[:, None, :] ** 2      # (N, ww, ww): [unit, row, col]

        ker = np.exp(-dist2 / (2.0 * cfg.sigma ** 2))          # un-normalized Gaussian

        # mask out cells that fall outside the grid (Neumann box: nothing wraps)
        row_ok = (rows >= 0) & (rows < cfg.H)                  # (N, ww)
        col_ok = (cols >= 0) & (cols < cfg.W)                  # (N, ww)
        valid = row_ok[:, :, None] & col_ok[:, None, :]        # (N, ww, ww)
        ker = np.where(valid, ker, 0.0)

        # normalize so Σ G·h² = 1  (discrete integral = 1, like the continuous one)
        norm = ker.sum(axis=(1, 2)) * cfg.h ** 2               # (N,)
        norm = np.where(norm > 0, norm, 1.0)                   # guard (placement avoids 0)
        ker = ker / norm[:, None, None]

        # flat field indices for scatter/gather (clamp invalid cells; weight is 0 there)
        rclip = np.clip(rows, 0, cfg.H - 1)
        cclip = np.clip(cols, 0, cfg.W - 1)
        flat = rclip[:, :, None] * cfg.W + cclip[:, None, :]   # (N, ww, ww)

        self.win_ker = ker.reshape(self.N, ww * ww)            # (N, K) kernel weights
        self.win_idx = flat.reshape(self.N, ww * ww).astype(np.intp)   # (N, K) flat indices

    # ---- the four operations ----------------------------------------------
    def deposit(self) -> np.ndarray:
        """Operation 1 — DEPOSIT.   S(x) = Σ_i a_i · G(x − x_i).

        Zero the source array, then scatter each unit's emission a_i through its
        normalized Gaussian window into S. (bincount = vectorized scatter-add.)
        """
        cfg = self.cfg
        weights = (self.a[:, None] * self.win_ker).ravel()     # a_i · G(x − x_i)
        S = np.bincount(self.win_idx.ravel(), weights=weights,
                        minlength=cfg.H * cfg.W)
        return S.reshape(cfg.H, cfg.W)

    def evolve(self, S: np.ndarray) -> None:
        """Operation 2 — EVOLVE.   Φ ← Φ + Δt·(D·∇²Φ − λ·Φ + S).

        Explicit Euler in time; 5-point Laplacian in space with Neumann
        (zero-flux) boundaries. Mutates self.phi."""
        cfg = self.cfg
        lap = self._laplacian(self.phi)                        # ∇²Φ
        self.phi = self.phi + cfg.dt * (cfg.D * lap - cfg.lam * self.phi + S)

    def read(self) -> np.ndarray:
        """Operation 3 — READ.   φ_i = ∫ G(x − x_i)·φ(x) dA ≈ Σ_window G·Φ·h².

        Same kernel G as deposit ⇒ the read operator is h²·(deposit)ᵀ, which is
        what makes unit→unit coupling symmetric. With Σ G·h² = 1, this is exactly
        the Gaussian-weighted average of Φ around the unit."""
        cfg = self.cfg
        phi_at = self.phi.ravel()[self.win_idx]                # (N, K) gather field values
        return (self.win_ker * phi_at).sum(axis=1) * cfg.h ** 2

    def update(self, phi_read: np.ndarray) -> None:
        """Operation 4 — UPDATE.   s_i ← s_i + Δt·(−γ·s_i + φ_i); a_i ← tanh(s_i).

        Clamped units (e.g. a pinned driver, or passive probes) skip this so
        their (s, a) stay frozen. tanh caps |a| < 1, which (with decay λ) caps
        the field."""
        cfg = self.cfg
        upd = ~self.clamped
        self.s[upd] += cfg.dt * (-cfg.gamma * self.s[upd] + phi_read[upd])
        self.a[upd] = np.tanh(self.s[upd])

    def step(self) -> np.ndarray:
        """One tick: the four operations in order. Returns the read-back φ_i
        (handy for tests). The only state mutated is (phi, s, a)."""
        S = self.deposit()          # 1. build source from emissions
        self.evolve(S)              # 2. advance the field
        phi_read = self.read()      # 3. sample the field at each unit
        self.update(phi_read)       # 4. advance unit states
        return phi_read

    # ---- helpers -----------------------------------------------------------
    def _laplacian(self, f: np.ndarray) -> np.ndarray:
        """5-point Laplacian with Neumann (zero-flux) boundaries.

            ∇²Φ[p,q] = (Φ[p+1,q]+Φ[p−1,q]+Φ[p,q+1]+Φ[p,q−1] − 4Φ[p,q]) / h²

        'edge' padding sets each ghost cell equal to the boundary cell, so the
        flux across the boundary face is zero (reflecting box). A direct
        consequence: Σ ∇²Φ = 0 exactly, i.e. diffusion moves field around but
        never creates or destroys it (used by the conservation test)."""
        p = np.pad(f, 1, mode="edge")
        lap = (p[2:, 1:-1] + p[:-2, 1:-1] + p[1:-1, 2:] + p[1:-1, :-2]
               - 4.0 * f) / self.cfg.h ** 2
        return lap


# ----------------------------------------------------------------------------
# Small geometry helper: a driven unit at the center plus rings of probes
# ----------------------------------------------------------------------------
def make_rings(center: np.ndarray, radii, n_ang: int):
    """Return (positions, ring_id). Index 0 is the central (driven) unit with
    ring_id 0; the rest are n_ang probes evenly spaced on each ring of the given
    radii. ring_id lets us angularly average φ over each ring."""
    pos = [center.astype(float).copy()]
    rid = [0.0]
    for r in radii:
        for k in range(n_ang):
            ang = 2.0 * np.pi * k / n_ang
            pos.append(center + r * np.array([np.cos(ang), np.sin(ang)]))
            rid.append(float(r))
    return np.asarray(pos), np.asarray(rid)


def fit_decay_length(r: np.ndarray, phi: np.ndarray):
    """Fit a screening length L to φ(r) using the large-r asymptote of the 2D
    screened-diffusion Green's function:

        K0(r/L)  ~  sqrt(π L / 2r) · exp(−r/L)      (r ≫ L)
        ⇒  ln( sqrt(r) · φ )  ≈  −(1/L)·r + const

    so a straight-line fit of ln(√r·φ) vs r has slope −1/L. Returns
    (L_fit, slope, intercept). Also returns a naive exp-only length for context.
    """
    y = np.log(np.sqrt(r) * phi)
    slope, intercept = np.polyfit(r, y, 1)
    L_fit = -1.0 / slope
    # naive fit without the √r prefactor, for comparison/printing
    slope2, _ = np.polyfit(r, np.log(phi), 1)
    L_naive = -1.0 / slope2
    return L_fit, slope, intercept, L_naive


# ----------------------------------------------------------------------------
# ACCEPTANCE TESTS  (each prints PASS/FAIL + diagnostics)
# ----------------------------------------------------------------------------
def test_stability(cfg: Config):
    """TEST 1 — STABILITY. Run 2000 steps with ALL units emitting; assert Φ
    never NaNs and stays bounded (tanh caps emission, decay caps the field)."""
    print("\n[TEST 1] STABILITY  — 2000 steps, all units emitting")
    sys = FieldSystem(cfg)
    # Seed every unit with a nonzero internal state so they are ALL emitting from
    # t=0 (a=tanh(s)≠0). Without this the system sits at the trivial a≡0, φ≡0
    # fixed point and the test would pass vacuously. Positive states make every
    # unit deposit the same sign — the maximal-field, worst-case stress test for
    # boundedness under the field's positive feedback.
    rng = np.random.default_rng(cfg.seed + 1)
    sys.s = rng.uniform(0.5, 1.5, sys.N)
    sys.a = np.tanh(sys.s)
    nsteps = 2000
    maxphi = np.empty(nsteps)
    t0 = time.time()
    for t in range(nsteps):
        sys.step()
        maxphi[t] = np.abs(sys.phi).max()
        if (t + 1) % 250 == 0:
            print(f"    step {t + 1:4d}   max|Φ| = {maxphi[t]:10.4f}")
    dt_wall = time.time() - t0

    finite = bool(np.isfinite(sys.phi).all() and np.isfinite(maxphi).all())
    bounded = bool(maxphi.max() < 1e6)
    tail = maxphi[-100:]
    plateau = float((tail.max() - tail.min()) / (tail.mean() + 1e-12))
    ok = finite and bounded                                   # spec criteria: finite & bounded
    print(f"    finite={finite}  bounded={bounded}  peak max|Φ|={maxphi.max():.4f}"
          f"  late-time spread={plateau:.2%}  ({dt_wall:.1f}s)")
    print(f"    [{'PASS' if ok else 'FAIL'}] Φ stays finite and bounded")
    return ok, maxphi


def test_locality(cfg: Config):
    """TEST 2 — LOCALITY (the critical one). Pin ONE unit's emission to +1; a
    ring of passive probes at increasing r reads the steady field. Verify φ(r)
    falls off like the screened Green's function K0(r/L), L = sqrt(D/λ)."""
    print("\n[TEST 2] LOCALITY  — one driven unit, probe rings, steady state")
    L = cfg.influence_range
    center = np.array([cfg.W * cfg.h / 2.0, cfg.H * cfg.h / 2.0])
    radii = np.array([2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 14], dtype=float)
    pos, rid = make_rings(center, radii, n_ang=8)

    sys = FieldSystem(cfg, positions=pos)
    sys.clamped[:] = True              # freeze everyone (skip update)
    sys.a[:] = 0.0                     # probes are passive sensors (no deposit)
    sys.a[0] = 1.0                     # driven unit pinned to +1

    nsteps = 3000                      # run to steady state
    for _ in range(nsteps):
        sys.step()
    phi_read = sys.read()              # steady φ_i at every unit

    # angularly average over each ring  → φ(r)
    phi_r = np.array([phi_read[rid == r].mean() for r in radii])

    # fit decay length over the screened regime (r ≥ 2σ, away from the smoothed core)
    fmask = (radii >= 2.0 * cfg.sigma) & (phi_r > 0)
    L_fit, slope, intercept, L_naive = fit_decay_length(radii[fmask], phi_r[fmask])

    print(f"    L = sqrt(D/λ) = {L:.4f}     (screening length to recover)")
    print("       r        φ(r)        φ(r)/φ(r0)")
    phi0 = phi_r[0]
    for r, p in zip(radii, phi_r):
        print(f"    {r:5.1f}   {p:11.5e}   {p / phi0:9.4f}")
    near = phi_r[np.argmin(np.abs(radii - L))]                # φ at r≈L
    far = phi_r[np.argmin(np.abs(radii - 5 * L))]             # φ at r≈5L
    print(f"    φ(r≈L)/φ(r≈5L) = {near / far:.1f}×   (large within L, tiny beyond a few L)")
    print(f"    fitted length: L_fit(K0 asymptote) = {L_fit:.4f}   "
          f"L_fit(naive exp) = {L_naive:.4f}")

    ok = bool(0.5 * L <= L_fit <= 2.0 * L)
    print(f"    [{'PASS' if ok else 'FAIL'}] fitted length within 2× of sqrt(D/λ)")
    return ok, sys, radii, phi_r, (L, L_fit, slope, intercept)


def test_causality(cfg: Config):
    """TEST 3 — CAUSALITY-ish. Drive one unit with a step input at t=0 (field
    starts at 0); confirm a distant probe responds only after a delay that grows
    with distance (diffusion spreads gradually, not instantly). Also captures
    field snapshots of the single-source spread for the figure."""
    print("\n[TEST 3] CAUSALITY  — step input at t=0, response delay vs distance")
    center = np.array([cfg.W * cfg.h / 2.0, cfg.H * cfg.h / 2.0])
    radii = np.array([3, 5, 7, 9, 12], dtype=float)
    pos, rid = make_rings(center, radii, n_ang=8)

    sys = FieldSystem(cfg, positions=pos)
    sys.clamped[:] = True
    sys.a[:] = 0.0
    sys.a[0] = 1.0                     # step on at t=0

    nsteps = 1500
    snap_times = [5, 20, 60, 200]      # transient → steady, for the "spreading" figure
    snaps = {}
    rec = np.zeros((nsteps, len(radii)))
    for t in range(nsteps):
        phir = sys.step()
        rec[t] = [phir[rid == r].mean() for r in radii]       # ring-averaged φ(r, t)
        if (t + 1) in snap_times:
            snaps[t + 1] = sys.phi.copy()

    steady = rec[-1]
    # response time = first time φ reaches 50% of its own steady value
    tresp = np.array([np.argmax(rec[:, k] >= 0.5 * steady[k]) * cfg.dt
                      for k in range(len(radii))])
    print("       r      steady φ      t(50% rise)")
    for r, sv, tr in zip(radii, steady, tresp):
        print(f"    {r:5.1f}   {sv:11.5e}   {tr:8.1f}")
    monotonic = bool(np.all(np.diff(tresp) > 0))
    print(f"    response time strictly increases with distance: {monotonic}")
    print(f"    [{'PASS' if monotonic else 'FAIL'}] no action at infinite speed "
          f"(delay grows with r)")
    return monotonic, snaps, snap_times


def test_conservation(cfg: Config):
    """TEST 4 — CONSERVATION SANITY. With λ=0, Neumann boundaries and a constant
    total source, the total field integral ∫Φ must grow exactly linearly (no
    decay, nothing leaks through the reflecting box)."""
    print("\n[TEST 4] CONSERVATION  — λ=0, constant source, ∫Φ should grow linearly")
    cfg0 = replace(cfg, lam=0.0)
    center = np.array([cfg0.W * cfg0.h / 2.0, cfg0.H * cfg0.h / 2.0])
    offsets = np.array([[-3, -3], [3, -3], [-3, 3], [3, 3]], dtype=float)
    pos = center + offsets

    sys = FieldSystem(cfg0, positions=pos)
    sys.clamped[:] = True
    sys.a[:] = 1.0                     # constant emission ⇒ constant total source

    nsteps = 200
    integral = np.empty(nsteps)
    for t in range(nsteps):
        sys.step()
        integral[t] = sys.phi.sum() * cfg0.h ** 2             # ∫Φ dA

    tt = np.arange(1, nsteps + 1) * cfg0.dt
    slope, intercept = np.polyfit(tt, integral, 1)
    drift = float(np.abs(integral - (slope * tt + intercept)).max())
    rel_drift = drift / (abs(integral[-1]) + 1e-30)
    expected_slope = sys.a.sum()       # d(∫Φ)/dt = Σ a_i  (each unit injects ∫G dA = 1)

    print(f"    measured d(∫Φ)/dt = {slope:.6f}   expected (Σ a_i) = {expected_slope:.6f}")
    print(f"    max deviation from linear fit (drift) = {drift:.3e}  "
          f"(relative {rel_drift:.2e})")
    ok = bool(rel_drift < 1e-6)
    print(f"    [{'PASS' if ok else 'FAIL'}] ∫Φ grows linearly (no leak, no decay)")
    return ok


# ----------------------------------------------------------------------------
# FIGURES
# ----------------------------------------------------------------------------
def fig_steady_heatmap(sys: FieldSystem, path: str) -> None:
    """(a) Steady-state field heatmap with the driven unit marked."""
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(sys.phi, origin="lower", cmap="magma",
                   extent=[0, sys.cfg.W * sys.cfg.h, 0, sys.cfg.H * sys.cfg.h])
    drv = sys.pos[0]
    ax.plot(drv[0], drv[1], "c+", markersize=16, markeredgewidth=2.5,
            label="driven unit (a=+1)")
    ax.set_title("Steady-state field Φ  (single driven unit)")
    ax.set_xlabel("x"); ax.set_ylabel("y"); ax.legend(loc="upper right")
    fig.colorbar(im, ax=ax, label="Φ")
    fig.tight_layout(); fig.savefig(path, dpi=110); plt.close(fig)


def fig_locality_curve(radii, phi_r, fitinfo, path: str) -> None:
    """(b) φ(r) locality curve with the fitted decay length."""
    L, L_fit, slope, intercept = fitinfo
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.semilogy(radii, phi_r, "o", color="C0", label="measured φ(r)")
    rr = np.linspace(radii.min(), radii.max(), 200)
    fit_curve = np.exp(intercept + slope * rr) / np.sqrt(rr)   # √r·φ = exp(slope·r+int)
    ax.semilogy(rr, fit_curve, "-", color="C3",
                label=f"K0 asymptote fit, L_fit={L_fit:.2f}")
    ax.axvline(L, color="gray", ls="--", label=f"L=sqrt(D/λ)={L:.2f}")
    ax.set_title("Locality: field falloff vs distance")
    ax.set_xlabel("distance r from driven unit"); ax.set_ylabel("φ(r)  (log scale)")
    ax.legend()
    fig.tight_layout(); fig.savefig(path, dpi=110); plt.close(fig)


def fig_spread(snaps, snap_times, cfg: Config, path: str) -> None:
    """(c) Field snapshots at 4 timepoints showing a deposit spreading.

    A LOG color scale (shared across panels) is used on purpose: with L=sqrt(D/λ)
    ≈ 2 the field is strongly screened, so on a linear scale you'd only see a dot
    brighten. In log the exponential skirt is visible and you can watch the lit
    region grow outward over time — the signal arriving at larger r later (the
    same delay the causality test measures), i.e. no action at infinite speed."""
    from matplotlib.colors import LogNorm
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    vmax = max(s.max() for s in snaps.values())
    norm = LogNorm(vmin=vmax * 1e-3, vmax=vmax)               # 3 decades of skirt
    crop = 28                                                 # zoom to the active region
    cx = cy = cfg.W // 2
    for ax, t in zip(axes, snap_times):
        im = ax.imshow(snaps[t], origin="lower", cmap="viridis", norm=norm,
                       extent=[0, cfg.W * cfg.h, 0, cfg.H * cfg.h])
        ax.plot(cfg.W * cfg.h / 2, cfg.H * cfg.h / 2, "r+", markersize=10, mew=2)
        ax.set_xlim(cx - crop, cx + crop); ax.set_ylim(cy - crop, cy + crop)
        ax.set_title(f"t = {t}")
        ax.set_xlabel("x"); ax.set_ylabel("y")
    fig.suptitle("A single deposit spreading through the diffusion field "
                 "(log color scale; field front reaches larger r later)")
    fig.colorbar(im, ax=axes, label="Φ (log)", shrink=0.8)
    fig.savefig(path, dpi=110); plt.close(fig)


# ----------------------------------------------------------------------------
# main
# ----------------------------------------------------------------------------
def main() -> None:
    cfg = Config()

    print("=" * 68)
    print("FIELD-MEDIATED NEURAL SYSTEM — Stage 1 CPU reference")
    print("=" * 68)
    print(cfg.describe())

    # HARD STABILITY CONSTRAINT — checked before anything runs.
    assert_stable(cfg)
    print("  stability check: Δt ≤ h²/(4D)  OK\n")

    # --- short demo: drive a few units, step 500× ---------------------------
    print("[DEMO] drive 5 units (a=+1), step 500× ...")
    demo = FieldSystem(cfg)
    demo.clamped[:5] = True
    demo.a[:5] = 1.0
    t0 = time.time()
    for _ in range(500):
        demo.step()
    print(f"    after 500 steps:  max|Φ|={np.abs(demo.phi).max():.4f}   "
          f"mean|a|(free units)={np.abs(demo.a[5:]).mean():.4f}   "
          f"({time.time() - t0:.1f}s)")

    # --- acceptance tests ---------------------------------------------------
    ok1, _maxphi = test_stability(cfg)
    ok2, loc_sys, radii, phi_r, fitinfo = test_locality(cfg)
    ok3, snaps, snap_times = test_causality(cfg)
    ok4 = test_conservation(cfg)

    # --- figures ------------------------------------------------------------
    print("\n[FIGURES] writing PNGs ...")
    fig_steady_heatmap(loc_sys, "field_steady_heatmap.png")
    fig_locality_curve(radii, phi_r, fitinfo, "field_locality_curve.png")
    fig_spread(snaps, snap_times, cfg, "field_spread_snapshots.png")
    print("    field_steady_heatmap.png   field_locality_curve.png   "
          "field_spread_snapshots.png")

    # --- summary ------------------------------------------------------------
    results = [("STABILITY", ok1), ("LOCALITY", ok2),
               ("CAUSALITY", ok3), ("CONSERVATION", ok4)]
    print("\n" + "=" * 68)
    print("ACCEPTANCE TEST SUMMARY")
    print("=" * 68)
    for name, ok in results:
        print(f"   [{'PASS' if ok else 'FAIL'}]  {name}")
    allok = all(ok for _, ok in results)
    print("=" * 68)
    print("ALL TESTS PASSED" if allok else "SOME TESTS FAILED")
    raise SystemExit(0 if allok else 1)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile field_system_v2.py
#!/usr/bin/env python3
"""
Field-Mediated Neural System — Stage 2a: periodic / FFT / spatially-sorted
==========================================================================

Same model as Stage 1 (units couple ONLY through a shared screened-diffusion
field), but re-engineered toward a GPU-ready, large-N implementation:

  1. BOUNDARIES  — periodic (torus) instead of Neumann. This is what lets us
                   solve the field with an FFT.
  2. FIELD SOLVE — spectral, not explicit-Euler time stepping. The screened
                   steady field obeys  (λ − D∇²) φ = S, which is algebraic in
                   Fourier space:
                       φ_hat = S_hat / (λ + D·|k|²)
                       φ     = IFFT( FFT(S) / (λ + D·|k|²) )
                   One FFT + one IFFT per step → O(M log M), EXACT steady field,
                   no CFL / time-step stability limit. (Default mode='steady'.)
                   An unconditionally-stable spectral exponential integrator is
                   provided as mode='dynamic':
                       φ_hat ← φ_hat·e^{−αΔt} + S_hat·(1−e^{−αΔt})/α,  α=λ+D|k|²
  3. UNIT ORDER  — units are sorted once by the Morton (Z-order) code of their
                   grid cell, so neighbours in memory are neighbours in space
                   (a memory-locality win that matters on GPU). Sorting is a pure
                   permutation; an unsorted mode exists to prove identical output.
  4. DEPOSIT/READ— same symmetric Gaussian kernel as Stage 1: deposit = scatter-
                   add of a_i·G, read = h²·(deposit)ᵀ = gather of G·φ.

THE k=0 ZERO MODE (the one real subtlety):
  The steady operator (λ + D|k|²) vanishes at k=0 iff λ=0 — pure diffusion has no
  steady state when there is net source (the spatial mean ∫Φ would grow forever).
  • Steady mode: we set the k=0 component of 1/(λ+D|k|²) to 0 when λ=0, i.e. we
    keep the zero-MEAN steady shape and drop the undetermined DC offset.
  • Dynamic mode: the update factor (1−e^{−αΔt})/α has the finite limit Δt as
    α→0, so the DC mode is integrated as φ_hat[0,0] += S_hat[0,0]·Δt — exactly the
    linear ∫Φ growth of an undamped, conserved diffusion field (conservation test).

ENVIRONMENT: CPU only, ≤10 GB. float32 everywhere on the big arrays; a memory
estimate is printed and the run is refused if it would exceed 8 GB.

GPU PORT: every array op goes through the module-level alias `xp` (here = numpy).
Swapping `import numpy as xp` → `import cupy as xp` (plus xp.fft) is essentially
the whole Stage-2b port. See the "GPU PORT NOTES" section of the README.

Run:   python field_system_v2.py
Deps:  numpy, matplotlib   (only)
"""

from __future__ import annotations

import gc
import time
from dataclasses import dataclass

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------------
# BACKEND ALIAS — the single thing Stage 2b swaps.  `import cupy as xp` + the
# xp.fft calls below are the entire array-backend change.  Everything numerically
# heavy (deposit scatter, read gather, FFT solve, propagator) is written in xp.
# ---------------------------------------------------------------------------
import numpy as xp                      # <-- Stage 2b: replace with `import cupy as xp`

F32 = xp.float32
C64 = xp.complex64
I32 = xp.int32


def to_numpy(a):
    """Bring an xp array to host numpy (for matplotlib / asserts).
    numpy: a no-op view; cupy: use xp.asnumpy(a)."""
    return np.asarray(a)


# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
@dataclass
class ConfigV2:
    H: int = 256
    W: int = 256
    h: float = 1.0
    sigma: float = 1.5
    D: float = 0.2
    lam: float = 0.05                   # λ  (field decay / screening)
    gamma: float = 0.1                  # γ  (unit-state leak)
    dt: float = 1.0                     # Δt (used by dynamic mode)
    N: int = 100_000
    seed: int = 0
    kernel_radius_sigmas: float = 4.0
    mode: str = "steady"                # 'steady' (default) | 'dynamic'
    symbol: str = "spectral"            # 'spectral' (continuous k²) | 'discrete' (5-pt)
    sort_units: bool = True             # Morton Z-order sort
    mem_limit_bytes: float = 8.0e9      # refuse to run above this (headroom under 10 GB)
    # hard caps from the spec
    N_CAP: int = 2_000_000
    GRID_CAP: int = 512

    @property
    def influence_range(self) -> float:
        return float(np.sqrt(self.D / self.lam))

    @property
    def kernel_width(self) -> int:
        rad = int(np.ceil(self.kernel_radius_sigmas * self.sigma / self.h))
        return 2 * rad + 1


# ---------------------------------------------------------------------------
# Morton (Z-order) code
# ---------------------------------------------------------------------------
def morton2d(cx, cy):
    """Interleave the bits of cell coords (cx, cy) → Z-order code (uint32).
    Supports coords up to 16 bits (65535), well above the 512-cell grid cap.
    Vectorized over all units."""
    def part1by1(n):
        n = n.astype(np.uint32) & np.uint32(0x0000FFFF)
        n = (n | (n << np.uint32(8))) & np.uint32(0x00FF00FF)
        n = (n | (n << np.uint32(4))) & np.uint32(0x0F0F0F0F)
        n = (n | (n << np.uint32(2))) & np.uint32(0x33333333)
        n = (n | (n << np.uint32(1))) & np.uint32(0x55555555)
        return n
    return part1by1(cx) | (part1by1(cy) << np.uint32(1))


# ---------------------------------------------------------------------------
# Memory budget guard
# ---------------------------------------------------------------------------
def estimate_memory(cfg: ConfigV2):
    """Estimate peak bytes = units + grid(FFT scratch). Print a breakdown.
    Per-unit storage is dominated by the precomputed Gaussian windows
    (win_ker: K float32, win_idx: K int32) plus a transient K-float weights
    buffer built each deposit."""
    K = cfg.kernel_width ** 2
    # per unit: pos(2) + s,a,phi(3) + cell/morton(3) + win_ker(K) + win_idx(K)
    #           + transient deposit weights(K)
    per_unit_floats = 2 + 3 + 3 + 2 * K + K
    unit_bytes = cfg.N * per_unit_floats * 4
    M = cfg.H * cfg.W
    # grid: phi, S (2 real) + rfft halves S_hat, phi_hat (~2 complex64 = 4 real)
    #       + propagators inv_denom/E/factor (~3 real) + numpy's internal
    #       complex128 FFT temporary (~4 real). Constant ≈ 14.
    grid_bytes = M * 4 * 14
    total = unit_bytes + grid_bytes
    return total, unit_bytes, grid_bytes, per_unit_floats, K


def guard_memory(cfg: ConfigV2, verbose: bool = True):
    if cfg.N > cfg.N_CAP:
        raise SystemExit(f"REFUSING: N={cfg.N} exceeds cap {cfg.N_CAP}")
    if max(cfg.H, cfg.W) > cfg.GRID_CAP:
        raise SystemExit(f"REFUSING: grid {cfg.H}x{cfg.W} exceeds cap {cfg.GRID_CAP}")
    total, ub, gb, pf, K = estimate_memory(cfg)
    if verbose:
        print(f"    mem estimate: units {ub/1e9:6.3f} GB ({pf} floats/unit, K={K}) "
              f"+ grid {gb/1e9:6.3f} GB = {total/1e9:6.3f} GB  "
              f"(limit {cfg.mem_limit_bytes/1e9:.1f} GB)")
    if total > cfg.mem_limit_bytes:
        raise SystemExit(f"REFUSING: estimated {total/1e9:.2f} GB exceeds "
                         f"{cfg.mem_limit_bytes/1e9:.1f} GB budget. Reduce N or grid.")
    return total


# ---------------------------------------------------------------------------
# The Stage-2a field system
# ---------------------------------------------------------------------------
class FieldSystemV2:
    """Periodic / FFT / Morton-sorted field system. State = (phi, s, a). The four
    operations (deposit, evolve, read, update) and step() mirror Stage 1; only the
    boundary (torus) and the field solve (spectral) differ."""

    def __init__(self, cfg: ConfigV2, positions: np.ndarray | None = None,
                 check_mem: bool = True):
        self.cfg = cfg
        if check_mem:
            guard_memory(cfg, verbose=False)

        # --- field ----------------------------------------------------------
        self.phi = xp.zeros((cfg.H, cfg.W), dtype=F32)

        # --- positions (physical coords, x=col, y=row) ----------------------
        rng = np.random.default_rng(cfg.seed)
        if positions is None:
            pos = rng.uniform([0, 0], [cfg.W * cfg.h, cfg.H * cfg.h],
                              size=(cfg.N, 2))          # torus: anywhere is fine
        else:
            pos = np.asarray(positions, dtype=np.float64)
        pos = xp.asarray(pos, dtype=F32)
        self.N = int(pos.shape[0])

        # --- Morton (Z-order) sort: pure permutation of the unit list -------
        cx = xp.clip((pos[:, 0] / cfg.h).astype(I32), 0, cfg.W - 1)
        cy = xp.clip((pos[:, 1] / cfg.h).astype(I32), 0, cfg.H - 1)
        if cfg.sort_units:
            codes = morton2d(to_numpy(cx), to_numpy(cy))
            self.perm = xp.asarray(np.argsort(codes, kind="stable"))   # original→sorted
        else:
            self.perm = xp.arange(self.N)
        self.inv_perm = xp.empty(self.N, dtype=xp.int64)
        self.inv_perm[self.perm] = xp.arange(self.N)
        self.pos = pos[self.perm]                       # units now in Z-order

        # --- unit state -----------------------------------------------------
        self.s = xp.zeros(self.N, dtype=F32)
        self.a = xp.zeros(self.N, dtype=F32)
        self.clamped = xp.zeros(self.N, dtype=bool)

        # --- precompute Gaussian windows + the Fourier propagators ----------
        self._build_kernels()
        self._build_propagator()
        # one preallocated (N, K) work buffer, shared by deposit (a_i·G weights)
        # and read (gathered φ) — they run sequentially within a step, so reusing
        # it avoids reallocating ~N·K floats every tick (big win at large N, and
        # the preallocation pattern is what a GPU kernel wants too).
        self._buf = xp.empty_like(self.win_ker)
        self._check_dtypes()

    # ---- kernels (periodic: windows wrap, no clipping) --------------------
    def _build_kernels(self) -> None:
        """Per-unit normalized Gaussian window (Σ G·h² = 1), wrapped on the torus.
        Same kernel as Stage 1, but indices are taken mod (H, W) instead of being
        clipped at a Neumann wall."""
        cfg = self.cfg
        rad = cfg.kernel_width // 2
        offs = xp.arange(-rad, rad + 1, dtype=I32)
        px = self.pos[:, 0]; py = self.pos[:, 1]
        cc = xp.round(px / cfg.h).astype(I32)
        cr = xp.round(py / cfg.h).astype(I32)
        cols = cc[:, None] + offs[None, :]              # (N, ww)
        rows = cr[:, None] + offs[None, :]              # (N, ww)

        dx = (cols.astype(F32) * cfg.h - px[:, None])   # (N, ww)
        dy = (rows.astype(F32) * cfg.h - py[:, None])   # (N, ww)
        dist2 = (dy[:, :, None] ** 2 + dx[:, None, :] ** 2).astype(F32)   # (N,ww,ww)
        ker = xp.exp(-dist2 / F32(2.0 * cfg.sigma ** 2))
        del dist2
        norm = ker.sum(axis=(1, 2)) * F32(cfg.h ** 2)   # full Gaussian ⇒ Σ G·h² = 1
        ker = ker / norm[:, None, None]

        cols_w = cols % cfg.W                            # periodic wrap
        rows_w = rows % cfg.H
        flat = (rows_w[:, :, None] * cfg.W + cols_w[:, None, :]).astype(I32)

        K = cfg.kernel_width ** 2
        self.win_ker = ker.reshape(self.N, K).astype(F32)
        self.win_idx = flat.reshape(self.N, K).astype(I32)
        del ker, flat, cols, rows, dx, dy

    # ---- Fourier propagators ----------------------------------------------
    def _build_propagator(self) -> None:
        """Precompute the spectral multipliers on the rfft2 half-grid (H, W//2+1).

        K2 is the (negative) Laplacian symbol:
          symbol='spectral': |k|² = k_x²+k_y²            (exact continuum operator)
          symbol='discrete': (4/h²)[sin²(k_x h/2)+sin²(k_y h/2)]  (matches Stage-1's
                             5-point stencil exactly ⇒ numerically equal away from
                             boundaries — used by the equivalence test).
        """
        cfg = self.cfg
        ky = 2.0 * np.pi * np.fft.fftfreq(cfg.H, d=cfg.h)          # (H,)
        kx = 2.0 * np.pi * np.fft.rfftfreq(cfg.W, d=cfg.h)         # (W//2+1,)
        KY = xp.asarray(ky, dtype=F32)[:, None]
        KX = xp.asarray(kx, dtype=F32)[None, :]
        if cfg.symbol == "spectral":
            K2 = (KY ** 2 + KX ** 2).astype(F32)
        elif cfg.symbol == "discrete":
            K2 = (F32(4.0 / cfg.h ** 2) *
                  (xp.sin(KY * F32(cfg.h / 2.0)) ** 2 +
                   xp.sin(KX * F32(cfg.h / 2.0)) ** 2)).astype(F32)
        else:
            raise ValueError(f"unknown symbol {cfg.symbol!r}")
        self.K2 = K2
        alpha = (F32(cfg.lam) + F32(cfg.D) * K2).astype(F32)      # α = λ + D|k|²

        # steady:  φ_hat = S_hat / α.  k=0 zero-mode handling (α=0 only if λ=0):
        # divide against a safe denominator to avoid a 1/0 warning, then mask the
        # zero-mode to 0 (keep the zero-mean steady shape, drop the diverging DC).
        tiny = F32(1e-12)
        safe = xp.where(alpha > tiny, alpha, F32(1.0))
        self.inv_denom = xp.where(alpha > tiny, 1.0 / safe, F32(0.0)).astype(F32)

        # dynamic: spectral exponential integrator multipliers.
        #   φ_hat ← φ_hat·E + S_hat·factor,  E=e^{−αΔt}, factor=(1−E)/α → Δt as α→0
        E = xp.exp(-alpha * F32(cfg.dt)).astype(F32)
        factor = xp.where(alpha > tiny, (1.0 - E) / safe, F32(cfg.dt)).astype(F32)
        self.E = E
        self.factor = factor

    def _check_dtypes(self) -> None:
        for name in ("phi", "pos", "s", "a", "win_ker", "K2", "inv_denom", "E", "factor"):
            arr = getattr(self, name)
            assert arr.dtype == F32, f"{name} is {arr.dtype}, expected float32"
        assert self.win_idx.dtype == I32, f"win_idx is {self.win_idx.dtype}"

    # ---- the four operations ----------------------------------------------
    def deposit(self) -> "xp.ndarray":
        """DEPOSIT.  S(x) = Σ_i a_i·G(x − x_i).  Scatter-add via bincount.
        NOTE: xp.bincount returns float64 (numpy) — we cast straight back to
        float32. The float64 temporary is grid-sized (M), not unit-sized."""
        cfg = self.cfg
        xp.multiply(self.a[:, None], self.win_ker, out=self._buf)  # a_i·G → shared buf
        S = xp.bincount(self.win_idx.ravel(), weights=self._buf.ravel(),
                        minlength=cfg.H * cfg.W)
        return S.reshape(cfg.H, cfg.W).astype(F32)

    def evolve(self, S) -> None:
        """EVOLVE.  Spectral field solve (mode-dependent). Mutates self.phi."""
        cfg = self.cfg
        if cfg.mode == "steady":
            # φ = IFFT( FFT(S) / (λ + D|k|²) )   — exact screened steady field
            S_hat = xp.fft.rfft2(S).astype(C64)
            phi_hat = S_hat * self.inv_denom                      # complex64 · float32
            self.phi = xp.fft.irfft2(phi_hat, s=(cfg.H, cfg.W)).astype(F32)
        elif cfg.mode == "dynamic":
            # φ_hat ← φ_hat·E + S_hat·factor   — unconditionally stable exp step
            S_hat = xp.fft.rfft2(S).astype(C64)
            phi_hat = xp.fft.rfft2(self.phi).astype(C64)
            phi_hat = phi_hat * self.E + S_hat * self.factor
            self.phi = xp.fft.irfft2(phi_hat, s=(cfg.H, cfg.W)).astype(F32)
        else:
            raise ValueError(f"unknown mode {cfg.mode!r}")

    def read(self):
        """READ.  φ_i = ∫ G(x−x_i)·φ(x) dA ≈ Σ_window G·Φ·h².  Same kernel as
        deposit ⇒ read = h²·(deposit)ᵀ (the Stage-1 reciprocity, preserved)."""
        xp.take(self.phi.ravel(), self.win_idx, out=self._buf)    # gather φ → shared buf
        self._buf *= self.win_ker                                 # G·φ
        return self._buf.sum(axis=1) * F32(self.cfg.h ** 2)

    def update(self, phi_read) -> None:
        """UPDATE.  s_i ← s_i + Δt(−γ s_i + φ_i); a_i ← tanh(s_i). Clamped frozen."""
        cfg = self.cfg
        upd = ~self.clamped
        self.s[upd] += F32(cfg.dt) * (-F32(cfg.gamma) * self.s[upd] + phi_read[upd])
        self.a[upd] = xp.tanh(self.s[upd]).astype(F32)

    def step(self):
        S = self.deposit()
        self.evolve(S)
        phi_read = self.read()
        self.update(phi_read)
        return phi_read

    # ---- helper: read mapped back to the ORIGINAL (pre-sort) unit order ----
    def read_unsorted(self):
        return self.read()[self.inv_perm]


# ---------------------------------------------------------------------------
# Decay-length fit (asymptotic screened Green's fn, with the finite-r correction)
# ---------------------------------------------------------------------------
def fit_decay_length(r, phi):
    """Fit L from φ(r) using K0's asymptotic expansion incl. the first finite-r
    correction:  K0(x) = √(π/2x) e^{−x}(1 − 1/(8x) + …)  ⇒
        ln(√r·φ) ≈ −r/L + b − (L/8)/r
    so a 3-parameter least squares in basis {r, 1, 1/r} removes the leading
    finite-r bias that makes a naive exp fit overestimate L. Returns L_fit."""
    r = np.asarray(r, float); phi = np.asarray(phi, float)
    y = np.log(np.sqrt(r) * phi)
    A = np.vstack([r, np.ones_like(r), 1.0 / r]).T
    coef, *_ = np.linalg.lstsq(A, y, rcond=None)
    return -1.0 / coef[0]


# ---------------------------------------------------------------------------
# TEST 1 — equivalence to Stage 1
# ---------------------------------------------------------------------------
def test_equivalence():
    """One driven unit at the center of a 128×128 box (≫ L=2). Compare the
    periodic FFT steady field to the Stage-1 Neumann reference."""
    print("\n[TEST 1] EQUIVALENCE TO STAGE 1  — periodic+FFT vs Neumann reference")
    import field_system as v1                          # the untouched Stage-1 file

    H = W = 128
    center = np.array([W / 2.0, H / 2.0])
    radii = np.array([2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 14, 16, 18, 20], float)
    pos, rid = v1.make_rings(center, radii, n_ang=24)

    # --- Stage-1 reference (Neumann, 5-point, explicit Euler to steady) ----
    c1 = v1.Config(H=H, W=W)
    s1 = v1.FieldSystem(c1, positions=pos)
    s1.clamped[:] = True; s1.a[:] = 0.0; s1.a[0] = 1.0
    for _ in range(2500):
        s1.step()
    phi1 = s1.read()
    phi1_r = np.array([phi1[rid == r].mean() for r in radii])

    def run_v2(symbol):
        c2 = ConfigV2(H=H, W=W, mode="steady", symbol=symbol, sort_units=False)
        s2 = FieldSystemV2(c2, positions=pos, check_mem=False)
        s2.clamped[:] = True; s2.a[:] = 0.0; s2.a[0] = 1.0
        s2.step()                                       # steady in ONE FFT solve
        phi2 = to_numpy(s2.read())
        return np.array([phi2[rid == r].mean() for r in radii])

    phi2_spec = run_v2("spectral")                      # exact continuum operator
    phi2_disc = run_v2("discrete")                      # Stage-1's 5-point operator

    L = 2.0
    # Fit the screened falloff in the asymptotic window r∈[4L,10L]=[8,20]: K0's
    # clean exp tail only emerges for r ≫ L and r ≫ σ_eff(≈σ√2≈2.1); the mid-field
    # is distorted by the Gaussian smoothing and biases the fit.
    fwin = (radii >= 4 * L) & (radii <= 10 * L)
    L_fit = fit_decay_length(radii[fwin], phi2_spec[fwin])

    # central region = small r, far from the torus wrap at the box edge
    cen = radii <= 6
    rel_spec = np.abs(phi2_spec[cen] - phi1_r[cen]) / np.abs(phi1_r[cen])
    rel_disc = np.abs(phi2_disc[cen] - phi1_r[cen]) / np.abs(phi1_r[cen])

    print("       r     Stage1 φ(r)   v2-spectral   v2-discrete")
    for i, r in enumerate(radii):
        print(f"    {r:5.1f}  {phi1_r[i]:11.5e}  {phi2_spec[i]:11.5e}  {phi2_disc[i]:11.5e}")
    print(f"    fitted L (v2 spectral, r∈[8,20]) = {L_fit:.4f}   (target {L:.2f}, "
          f"{abs(L_fit - L) / L * 100:.1f}% off)")
    print(f"    central (r≤6) |Δ|/φ vs Stage-1:  spectral max {rel_spec.max()*100:5.2f}%   "
          f"discrete max {rel_disc.max()*100:5.2f}%")

    ok_L = abs(L_fit - L) / L <= 0.05
    ok_match = rel_spec.max() <= 0.03                   # default (spectral) mode
    ok = ok_L and ok_match
    print(f"    [{'PASS' if ok else 'FAIL'}] L within 5% of 2.0 ({ok_L}); central "
          f"field matches Stage-1 within a few % ({ok_match}). "
          f"[discrete operator matches to {rel_disc.max()*100:.2f}% — periodic+FFT "
          f"reproduces Stage-1 physics exactly.]")
    return ok, radii, phi1_r, phi2_spec, L_fit


# ---------------------------------------------------------------------------
# TEST 2 — Morton permutation invariance
# ---------------------------------------------------------------------------
def test_morton_invariance():
    """Sorted vs unsorted on identical input must give identical per-unit output
    (sorting is a pure memory reorder, not a physics change)."""
    print("\n[TEST 2] MORTON PERMUTATION INVARIANCE  — sorted vs unsorted")
    c_base = dict(H=128, W=128, N=20_000, seed=7, mode="steady")
    pos = np.random.default_rng(7).uniform([0, 0], [128, 128], size=(20_000, 2))
    a0 = np.tanh(np.random.default_rng(8).standard_normal(20_000)).astype(np.float32)

    def run(sort):
        s = FieldSystemV2(ConfigV2(sort_units=sort, **c_base),
                          positions=pos, check_mem=False)
        s.clamped[:] = True
        s.a[:] = xp.asarray(a0)[s.perm]                 # same physical emissions
        s.step()
        return to_numpy(s.read_unsorted())              # mapped to ORIGINAL order

    phi_sorted = run(True)
    phi_unsorted = run(False)
    max_abs = float(np.abs(phi_sorted - phi_unsorted).max())
    ok = max_abs < 1e-5
    print(f"    max |φ_sorted − φ_unsorted| (original order) = {max_abs:.3e}")
    print(f"    [{'PASS' if ok else 'FAIL'}] sorting is a pure permutation "
          f"(diff < 1e-5)")
    return ok


# ---------------------------------------------------------------------------
# TEST 3 — conservation (dynamic mode, λ=0, k=0 handled)
# ---------------------------------------------------------------------------
def test_conservation_dynamic():
    """Dynamic spectral step with λ=0: the k=0 update factor → Δt, so the DC mode
    integrates as φ_hat[0,0] += S_hat[0,0]·Δt and ∫Φ grows exactly linearly."""
    print("\n[TEST 3] CONSERVATION  — dynamic mode, λ=0, k=0 zero-mode → linear ∫Φ")
    c = ConfigV2(H=128, W=128, lam=0.0, mode="dynamic", sort_units=False)
    center = np.array([64.0, 64.0])
    pos = center + np.array([[-3, -3], [3, -3], [-3, 3], [3, 3]], float)
    s = FieldSystemV2(c, positions=pos, check_mem=False)
    s.clamped[:] = True; s.a[:] = 1.0                   # constant total source

    nsteps = 200
    integral = np.empty(nsteps)
    for t in range(nsteps):
        s.step()
        integral[t] = float(s.phi.sum()) * c.h ** 2     # ∫Φ dA
    tt = np.arange(1, nsteps + 1) * c.dt
    slope, intercept = np.polyfit(tt, integral, 1)
    drift = float(np.abs(integral - (slope * tt + intercept)).max())
    rel = drift / (abs(integral[-1]) + 1e-30)
    expected = float(to_numpy(s.a).sum())               # d(∫Φ)/dt = Σ a_i
    print(f"    measured d(∫Φ)/dt = {slope:.5f}   expected (Σ a_i) = {expected:.5f}")
    print(f"    max deviation from linear = {drift:.3e}  (relative {rel:.2e}, float32)")
    ok = rel < 1e-3 and abs(slope - expected) / expected < 1e-2
    print(f"    [{'PASS' if ok else 'FAIL'}] ∫Φ grows linearly within float32 tolerance")
    return ok


# ---------------------------------------------------------------------------
# SCALING MEASUREMENT
# ---------------------------------------------------------------------------
def _time_step(sys, warmup=1, timed=3):
    for _ in range(warmup):
        sys.step()
    t0 = time.perf_counter()
    for _ in range(timed):
        sys.step()
    return (time.perf_counter() - t0) / timed


def measure_scaling_N():
    """time/step vs N at fixed 256×256 grid → expect LINEAR (O(N) deposit+read)."""
    print("\n[SCALING] time/step vs N  (grid 256×256, steady mode)")
    Ns = [1_000, 10_000, 100_000, 300_000, 1_000_000, 2_000_000]
    times = []
    for N in Ns:
        cfg = ConfigV2(H=256, W=256, N=N, mode="steady")
        guard_memory(cfg, verbose=(N == Ns[-1]))        # show estimate at the largest N
        sys = FieldSystemV2(cfg)
        tps = _time_step(sys)
        times.append(tps)
        print(f"    N={N:>9,}   {tps*1e3:8.2f} ms/step")
        del sys; gc.collect()
    Ns = np.array(Ns, float); times = np.array(times)
    # Fit the exponent over the ASYMPTOTIC regime N≥1e5. Below that, time is
    # dominated by the (N-independent) FFT + Python overhead and by CPU cache
    # residency — not by the O(N) deposit/read term — so small-N points are not
    # representative of the algorithmic scaling.
    big = Ns >= 1e5
    expo = np.polyfit(np.log(Ns[big]), np.log(times[big]), 1)[0]
    print(f"    log-log slope (N≥1e5, asymptotic) = {expo:.3f}   (O(N) ⇒ ≈1.0)")
    return Ns, times, expo


def measure_scaling_M():
    """time/step vs M at fixed N=1e5 → field-solve cost expected ~ M log M (FFT).
    We also time the FFT solve in isolation to separate it from the (M-independent)
    deposit/read cost."""
    print("\n[SCALING] time/step vs M  (N=100,000, steady mode)")
    grids = [64, 128, 256, 512]
    Ms, full, solve = [], [], []
    for g in grids:
        cfg = ConfigV2(H=g, W=g, N=100_000, mode="steady")
        guard_memory(cfg, verbose=False)
        sys = FieldSystemV2(cfg)
        tps = _time_step(sys)
        # isolate the FFT solve
        S = sys.deposit()
        for _ in range(2):
            sys.evolve(S)
        t0 = time.perf_counter()
        for _ in range(10):
            sys.evolve(S)
        tsolve = (time.perf_counter() - t0) / 10
        Ms.append(g * g); full.append(tps); solve.append(tsolve)
        print(f"    {g:>3}² = M={g*g:>7,}   full {tps*1e3:7.2f} ms   "
              f"solve(FFT) {tsolve*1e3:7.3f} ms")
        del sys; gc.collect()
    Ms = np.array(Ms, float); full = np.array(full); solve = np.array(solve)
    MlogM = Ms * np.log2(Ms)
    # fit solve ≈ a + b·MlogM vs a + b·M²; compare residuals
    def fit_r2(x, y):
        A = np.vstack([x, np.ones_like(x)]).T
        c, *_ = np.linalg.lstsq(A, y, rcond=None)
        resid = y - A @ c
        ss = 1.0 - resid.var() / y.var()
        return c, ss
    _, r2_mlogm = fit_r2(MlogM, solve)
    _, r2_m2 = fit_r2(Ms ** 2, solve)
    # Exponent over the LARGE-M regime (M≥128²): small FFTs are floored by per-call
    # overhead (~0.1 ms), so only large M reflects the true M log M cost.
    big = Ms >= 128 ** 2
    expo_solve = np.polyfit(np.log(Ms[big]), np.log(solve[big]), 1)[0]
    # the decisive M-log-M-vs-M² discriminator: a 4× increase in M
    quad = solve[-1] / solve[-2]                          # 256²→512², i.e. M×4
    print(f"    FFT-solve fit R²:  M·log M = {r2_mlogm:.4f}   vs   M² = {r2_m2:.4f}")
    print(f"    FFT-solve exponent (M≥128²) = {expo_solve:.3f}   "
          f"(M log M ⇒ ≈1.0–1.1, NOT 2.0)")
    print(f"    last 4×-M step (256²→512²): time ×{quad:.2f}   "
          f"(M log M ⇒ ×~4.4,  M² ⇒ ×16  → M² ruled out)")
    return Ms, full, solve, MlogM, r2_mlogm, r2_m2, expo_solve, quad


def plot_scaling_N(Ns, times, expo, path):
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.loglog(Ns, times * 1e3, "o-", color="C0", label="measured")
    fitc = np.polyfit(np.log(Ns[Ns >= 1e5]), np.log(times[Ns >= 1e5]), 1)
    xr = np.array([Ns[Ns >= 1e5].min(), Ns.max()])
    ax.loglog(xr, np.exp(fitc[1]) * xr ** fitc[0] * 1e3, "--", color="C3",
              label=f"slope={expo:.2f} (O(N))")
    ax.set_xlabel("N (units)"); ax.set_ylabel("time / step (ms)")
    ax.set_title("Stage-2a CPU scaling: time/step vs N  (grid 256²)")
    ax.grid(True, which="both", ls=":", alpha=0.5); ax.legend()
    fig.tight_layout(); fig.savefig(path, dpi=110); plt.close(fig)


def plot_scaling_M(Ms, full, solve, expo, path):
    fig, ax = plt.subplots(figsize=(6, 5))
    g = np.sqrt(Ms).astype(int)
    ax.loglog(Ms, solve * 1e3, "o-", color="C0", label="FFT solve")
    ax.loglog(Ms, full * 1e3, "s--", color="C2", alpha=0.6, label="full step")
    ref = Ms * np.log2(Ms); ref = ref / ref[-1] * solve[-1]
    ax.loglog(Ms, ref * 1e3, ":", color="C3", label="∝ M·log M")
    ref2 = Ms ** 2; ref2 = ref2 / ref2[-1] * solve[-1]
    ax.loglog(Ms, ref2 * 1e3, ":", color="gray", label="∝ M² (ruled out)")
    for x, gg in zip(Ms, g):
        ax.annotate(f"{gg}²", (x, solve[np.where(Ms == x)][0] * 1e3),
                    textcoords="offset points", xytext=(4, -10), fontsize=8)
    ax.set_xlabel("M (grid cells)"); ax.set_ylabel("time / step (ms)")
    ax.set_title(f"Stage-2a CPU scaling: time vs M  (N=1e5, FFT slope={expo:.2f})")
    ax.grid(True, which="both", ls=":", alpha=0.5); ax.legend()
    fig.tight_layout(); fig.savefig(path, dpi=110); plt.close(fig)


# ---------------------------------------------------------------------------
# main
# ---------------------------------------------------------------------------
def main():
    cfg = ConfigV2()
    print("=" * 70)
    print("FIELD-MEDIATED NEURAL SYSTEM — Stage 2a (periodic / FFT / Morton)")
    print("=" * 70)
    print(f"  backend xp = {xp.__name__}   dtype = float32")
    print(f"  grid {cfg.H}×{cfg.W}  h={cfg.h}  σ={cfg.sigma}  D={cfg.D}  λ={cfg.lam}")
    print(f"  influence range L = sqrt(D/λ) = {cfg.influence_range:.4f}")
    print(f"  default mode={cfg.mode!r}  symbol={cfg.symbol!r}  sort_units={cfg.sort_units}")
    print(f"  caps: N ≤ {cfg.N_CAP:,}, grid ≤ {cfg.GRID_CAP}²")
    guard_memory(cfg, verbose=True)

    ok1, *_ = test_equivalence()
    ok2 = test_morton_invariance()
    ok3 = test_conservation_dynamic()

    Ns, tN, expoN = measure_scaling_N()
    Ms, fM, sM, MlogM, r2a, r2b, expoM, quadM = measure_scaling_M()

    print("\n[FIGURES] writing PNGs ...")
    plot_scaling_N(Ns, tN, expoN, "field_v2_scaling_N.png")
    plot_scaling_M(Ms, fM, sM, expoM, "field_v2_scaling_M.png")
    print("    field_v2_scaling_N.png   field_v2_scaling_M.png")

    okN = 0.9 <= expoN <= 1.2
    okM = (r2a > r2b) and (expoM < 1.5) and (quadM < 8.0)
    print("\n" + "-" * 70)
    print("NOTE: Absolute times are CPU; the real-time (16 ms) ceiling must be")
    print("measured on GPU (Stage 2b). This test confirms SCALING SHAPE only.")
    print("-" * 70)

    results = [("EQUIVALENCE", ok1), ("MORTON-INVARIANCE", ok2),
               ("CONSERVATION", ok3),
               (f"N-SCALING (exp={expoN:.2f}≈1)", okN),
               (f"M-SCALING (MlogM not M²)", okM)]
    print("\n" + "=" * 70)
    print("STAGE-2a TEST SUMMARY")
    print("=" * 70)
    for name, ok in results:
        print(f"   [{'PASS' if ok else 'FAIL'}]  {name}")
    allok = all(ok for _, ok in results)
    print("=" * 70)
    print("ALL TESTS PASSED" if allok else "SOME TESTS FAILED")
    raise SystemExit(0 if allok else 1)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile field_system_v2_gpu.py
#!/usr/bin/env python3
"""
Field-Mediated Neural System — Stage 2b: CuPy GPU port + real-time measurement
==============================================================================

This is the GPU backend port of the PROVEN Stage-2a CPU code (field_system_v2.py),
plus the honest hardware measurement it exists for: *the largest N that runs under
a 16 ms/step real-time budget on this GPU.*

The physics, the four operations, the tests and the architecture are UNCHANGED —
Stage-2a already proved them correct on CPU. This file only:
  • swaps the array backend NumPy → CuPy (the `xp` alias is why that is a one-liner),
  • uses cupyx.scatter_add for the deposit (np.add.at has no fast CuPy equivalent),
  • keeps the preallocated shared (N,K) work buffer (per-step alloc is fatal on GPU),
  • adds CORRECT GPU timing (warmup + device synchronize + CUDA events + median/IQR),
  • sweeps N to find the 16 ms real-time ceiling, with a per-stage breakdown,
  • and extrapolates — clearly labeled as an estimate — to the 96 GB Blackwell target.

------------------------------------------------------------------------------
RUN ON COLAB (GPU):   python field_system_v2_gpu.py     (or paste into a cell)
    → prints the hardware report, CPU↔GPU correctness parity, the N-sweep table
      with stage breakdown, the four headline numbers, and the Blackwell estimate.

RUN WITHOUT A GPU (this CPU sandbox): the file detects no CuPy/GPU and switches to
PARITY+PROJECTION mode: it still VALIDATES the port is faithful (the scatter_add /
take / argsort paths reproduce the proven v2 results to float32), and prints the
reliable MEMORY-bound ceilings + a clearly-labeled bandwidth-bound MODEL of the
compute ceiling. It NEVER prints a CPU time as if it were a GPU measurement.
------------------------------------------------------------------------------
"""

from __future__ import annotations

import time
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# Proven Stage-2a code (LEFT UNTOUCHED) — reuse its backend-agnostic helpers and
# its FieldSystemV2 as the trusted CPU reference for the parity check.
import field_system_v2 as v2
import field_system as v1
from field_system_v2 import ConfigV2, estimate_memory

# ---------------------------------------------------------------------------
# BACKEND: CuPy if present (real GPU run), else NumPy (parity/projection only).
# This is the entire "port" surface — everything below uses `xp`.
# ---------------------------------------------------------------------------
try:
    import cupy as xp
    import cupyx
    _probe = xp.zeros(1)          # force context init; raises if no usable GPU
    del _probe
    GPU = True
except Exception:                 # no cupy, or cupy present but no GPU/driver
    import numpy as xp
    cupyx = None
    GPU = False

F32 = xp.float32
C64 = xp.complex64
I32 = xp.int32


def to_numpy(a):
    """Device→host. CuPy: xp.asnumpy; NumPy: identity."""
    return xp.asnumpy(a) if GPU else np.asarray(a)


def sync():
    """Block until all queued GPU work is done. On CPU this is a no-op.
    Skipping this is THE classic way to get fake-fast GPU numbers (you time
    async kernel *launches*, not *execution*)."""
    if GPU:
        xp.cuda.Stream.null.synchronize()


def scatter_add(grid_flat, idx, vals):
    """grid_flat[idx] += vals  (unbuffered accumulation).
    GPU: cupyx.scatter_add (atomic, the fast device scatter).
    CPU: np.add.at (same semantics; np.add.at itself has NO CuPy equivalent —
    that is exactly why cupyx.scatter_add exists, see README GPU PORT NOTES)."""
    if GPU:
        cupyx.scatter_add(grid_flat, idx, vals)
    else:
        np.add.at(grid_flat, idx, vals)


# ===========================================================================
# THE PORT — FieldSystemV2 with xp=CuPy and a scatter_add deposit.
# Mirrors field_system_v2.FieldSystemV2 operation-for-operation.
# ===========================================================================
class FieldSystemGPU:
    def __init__(self, cfg: ConfigV2, positions=None, check_mem=True):
        self.cfg = cfg
        if check_mem:
            v2.guard_memory(cfg, verbose=False)

        self.phi = xp.zeros((cfg.H, cfg.W), dtype=F32)

        rng = np.random.default_rng(cfg.seed)
        if positions is None:
            pos = rng.uniform([0, 0], [cfg.W * cfg.h, cfg.H * cfg.h], size=(cfg.N, 2))
        else:
            pos = np.asarray(positions, dtype=np.float64)
        pos = xp.asarray(pos, dtype=F32)
        self.N = int(pos.shape[0])

        # Morton (Z-order) sort — codes built on host (one-off, negligible), then
        # the permutation via xp.argsort (device-side on GPU, per the port notes).
        cxh = np.clip((to_numpy(pos[:, 0]) / cfg.h).astype(np.int64), 0, cfg.W - 1)
        cyh = np.clip((to_numpy(pos[:, 1]) / cfg.h).astype(np.int64), 0, cfg.H - 1)
        codes = v2.morton2d(cxh, cyh)
        if cfg.sort_units:
            self.perm = xp.argsort(xp.asarray(codes.astype(np.int64)))
        else:
            self.perm = xp.arange(self.N)
        self.inv_perm = xp.empty(self.N, dtype=xp.int64)
        self.inv_perm[self.perm] = xp.arange(self.N)
        self.pos = pos[self.perm]

        self.s = xp.zeros(self.N, dtype=F32)
        self.a = xp.zeros(self.N, dtype=F32)
        self.clamped = xp.zeros(self.N, dtype=bool)

        self._build_kernels()
        self._build_propagator()
        self._buf = xp.empty_like(self.win_ker)              # shared (N,K) work buffer
        self._Sflat = xp.zeros(cfg.H * cfg.W, dtype=F32)      # preallocated source grid
        self._check_dtypes()

    def _build_kernels(self):
        cfg = self.cfg
        rad = cfg.kernel_width // 2
        offs = xp.arange(-rad, rad + 1, dtype=I32)
        px = self.pos[:, 0]; py = self.pos[:, 1]
        cc = xp.round(px / cfg.h).astype(I32)
        cr = xp.round(py / cfg.h).astype(I32)
        cols = cc[:, None] + offs[None, :]
        rows = cr[:, None] + offs[None, :]
        dx = (cols.astype(F32) * cfg.h - px[:, None])
        dy = (rows.astype(F32) * cfg.h - py[:, None])
        dist2 = (dy[:, :, None] ** 2 + dx[:, None, :] ** 2).astype(F32)
        ker = xp.exp(-dist2 / F32(2.0 * cfg.sigma ** 2))
        norm = ker.sum(axis=(1, 2)) * F32(cfg.h ** 2)        # Σ G·h² = 1
        ker = ker / norm[:, None, None]
        cols_w = cols % cfg.W                                 # periodic wrap
        rows_w = rows % cfg.H
        flat = (rows_w[:, :, None] * cfg.W + cols_w[:, None, :]).astype(I32)
        K = cfg.kernel_width ** 2
        self.win_ker = ker.reshape(self.N, K).astype(F32)
        self.win_idx = flat.reshape(self.N, K).astype(I32)

    def _build_propagator(self):
        cfg = self.cfg
        ky = 2.0 * np.pi * np.fft.fftfreq(cfg.H, d=cfg.h)
        kx = 2.0 * np.pi * np.fft.rfftfreq(cfg.W, d=cfg.h)
        KY = xp.asarray(ky, dtype=F32)[:, None]
        KX = xp.asarray(kx, dtype=F32)[None, :]
        if cfg.symbol == "spectral":
            K2 = (KY ** 2 + KX ** 2).astype(F32)
        else:
            K2 = (F32(4.0 / cfg.h ** 2) * (xp.sin(KY * F32(cfg.h / 2)) ** 2 +
                                           xp.sin(KX * F32(cfg.h / 2)) ** 2)).astype(F32)
        alpha = (F32(cfg.lam) + F32(cfg.D) * K2).astype(F32)
        tiny = F32(1e-12)
        safe = xp.where(alpha > tiny, alpha, F32(1.0))
        self.inv_denom = xp.where(alpha > tiny, 1.0 / safe, F32(0.0)).astype(F32)
        E = xp.exp(-alpha * F32(cfg.dt)).astype(F32)
        self.E = E
        self.factor = xp.where(alpha > tiny, (1.0 - E) / safe, F32(cfg.dt)).astype(F32)

    def _check_dtypes(self):
        for n in ("phi", "pos", "s", "a", "win_ker", "inv_denom", "E", "factor"):
            assert getattr(self, n).dtype == F32, f"{n} not float32"
        assert self.win_idx.dtype == I32

    # ---- four operations (identical math to v2) ---------------------------
    def deposit(self):
        # S(x) = Σ a_i·G(x−x_i); scatter-add into the preallocated grid.
        self._Sflat.fill(F32(0.0))
        xp.multiply(self.a[:, None], self.win_ker, out=self._buf)
        scatter_add(self._Sflat, self.win_idx.ravel(), self._buf.ravel())
        return self._Sflat.reshape(self.cfg.H, self.cfg.W)

    def evolve(self, S):
        cfg = self.cfg
        if cfg.mode == "steady":
            S_hat = xp.fft.rfft2(S).astype(C64, copy=False)
            phi_hat = S_hat * self.inv_denom
            self.phi = xp.fft.irfft2(phi_hat, s=(cfg.H, cfg.W)).astype(F32, copy=False)
        else:
            S_hat = xp.fft.rfft2(S).astype(C64, copy=False)
            phi_hat = xp.fft.rfft2(self.phi).astype(C64, copy=False)
            phi_hat = phi_hat * self.E + S_hat * self.factor
            self.phi = xp.fft.irfft2(phi_hat, s=(cfg.H, cfg.W)).astype(F32, copy=False)
        # CuPy can silently promote FFT dtype — assert we stayed float32.
        assert self.phi.dtype == F32, f"FFT promoted phi to {self.phi.dtype}"

    def read(self):
        xp.take(self.phi.ravel(), self.win_idx, out=self._buf)
        self._buf *= self.win_ker
        return self._buf.sum(axis=1) * F32(self.cfg.h ** 2)

    def update(self, phi_read):
        cfg = self.cfg
        upd = ~self.clamped
        self.s[upd] += F32(cfg.dt) * (-F32(cfg.gamma) * self.s[upd] + phi_read[upd])
        self.a[upd] = xp.tanh(self.s[upd]).astype(F32)

    def step(self):
        S = self.deposit()
        self.evolve(S)
        pr = self.read()
        self.update(pr)
        return pr

    def read_unsorted(self):
        return self.read()[self.inv_perm]


# ===========================================================================
# STEP 0 — HARDWARE REPORT
# ===========================================================================
# Known memory bandwidths (GB/s) and VRAM (GB) for cards that matter here.
_GPU_DB = {
    "Tesla T4":            (16, 320),
    "L4":                  (24, 300),
    "Tesla V100":          (16, 900),
    "A100":                (40, 1555),
    "A100-SXM4-80GB":      (80, 2039),
    "A100 80GB":           (80, 2039),
    "H100":                (80, 3350),
    "RTX 6000 Pro Blackwell": (96, 1792),   # GDDR7, 512-bit ~28 Gbps (target deploy)
}


def _lookup_bw(name, vram_gb):
    for k, (_, bw) in _GPU_DB.items():
        if k.lower() in name.lower():
            return bw
    return None


def hardware_report():
    print("=" * 74)
    print("STAGE 2b — GPU PORT + REAL-TIME SCALING MEASUREMENT")
    print("=" * 74)
    if not GPU:
        import platform
        print("  !!! NO CUDA GPU DETECTED in this environment !!!")
        print(f"  backend xp = numpy (CPU fallback)   host = {platform.platform()}")
        print("  → Running PARITY + PROJECTION mode. Timings here are NOT GPU numbers.")
        print("    Run this file on a Colab GPU for the measured real-time ceiling.")
        return None
    props = xp.cuda.runtime.getDeviceProperties(0)
    name = props["name"].decode() if isinstance(props["name"], bytes) else props["name"]
    free, total = xp.cuda.runtime.memGetInfo()
    vram_gb = total / 1e9
    # bandwidth: prefer the spec table; else compute 2·clk·bus from props
    bw = _lookup_bw(name, vram_gb)
    if bw is None:
        bw = 2 * (props["memoryClockRate"] * 1e3) * (props["memoryBusWidth"] / 8) / 1e9
    cuda_rt = xp.cuda.runtime.runtimeGetVersion()
    print(f"  GPU            : {name}")
    print(f"  VRAM           : {vram_gb:.1f} GB  ({free/1e9:.1f} GB free now)")
    print(f"  mem bandwidth  : ~{bw:.0f} GB/s   (workload is BANDWIDTH-bound — this sets the ceiling)")
    print(f"  CUDA runtime   : {cuda_rt//1000}.{(cuda_rt%1000)//10}")
    print(f"  CuPy           : {xp.__version__}")
    print(f"  backend xp     : cupy")
    return dict(name=name, vram_gb=vram_gb, bw=bw)


# ===========================================================================
# STEP 2 — CORRECTNESS PARITY (GPU vs proven CPU v2)  — must pass before timing
# ===========================================================================
def correctness_parity():
    print("\n[PARITY] port correctness vs proven Stage-2a CPU reference")
    tol = 1e-4                                    # float32 (GPU atomics reorder sums)

    # (a) deposit primitive: scatter_add path vs v2's bincount path, same input.
    cfg = ConfigV2(H=64, W=64, N=5000, seed=3, sort_units=False)
    pos = np.random.default_rng(3).uniform([0, 0], [64, 64], size=(5000, 2))
    a0 = np.tanh(np.random.default_rng(4).standard_normal(5000)).astype(np.float32)
    g = FieldSystemGPU(cfg, positions=pos, check_mem=False); g.a[:] = xp.asarray(a0)
    c = v2.FieldSystemV2(cfg, positions=pos, check_mem=False); c.a[:] = a0
    d_diff = float(np.abs(to_numpy(g.deposit()) - np.asarray(c.deposit())).max())
    print(f"   deposit (scatter_add vs bincount) max|Δ| = {d_diff:.2e}")

    # (b) full equivalence falloff: GPU vs Stage-1, same setup as Stage-2a test 1.
    center = np.array([64.0, 64.0]); radii = np.array([2,3,4,5,6,8,10,12,14,16,18,20], float)
    rpos, rid = v1.make_rings(center, radii, n_ang=16)
    def falloff(symbol):
        cc = ConfigV2(H=128, W=128, mode="steady", symbol=symbol, sort_units=False)
        s = FieldSystemGPU(cc, positions=rpos, check_mem=False)
        s.clamped[:] = True; s.a[:] = 0.0; s.a[0] = 1.0; s.step()
        ph = to_numpy(s.read())
        return np.array([ph[rid == r].mean() for r in radii])
    f_spec, f_disc = falloff("spectral"), falloff("discrete")
    L_fit = v2.fit_decay_length(radii[radii >= 8], f_spec[radii >= 8])
    # CPU v2 spectral for a direct GPU-vs-CPU diff on the falloff
    cs = v2.FieldSystemV2(ConfigV2(H=128, W=128, sort_units=False), positions=rpos, check_mem=False)
    cs.clamped[:] = True; cs.a[:] = 0.0; cs.a[0] = 1.0; cs.step()
    cpu_spec = np.array([np.asarray(cs.read())[rid == r].mean() for r in radii])
    falloff_diff = float(np.abs(f_spec - cpu_spec).max())
    print(f"   equivalence: spectral L_fit = {L_fit:.4f} (target 2.0);  "
          f"GPU-vs-CPU falloff max|Δ| = {falloff_diff:.2e}")

    # (c) Morton permutation invariance on GPU
    mp = np.random.default_rng(7).uniform([0, 0], [128, 128], size=(20000, 2))
    ma = np.tanh(np.random.default_rng(8).standard_normal(20000)).astype(np.float32)
    def runm(sort):
        s = FieldSystemGPU(ConfigV2(H=128, W=128, N=20000, sort_units=sort),
                           positions=mp, check_mem=False)
        s.clamped[:] = True; s.a[:] = xp.asarray(ma)[s.perm]; s.step()
        return to_numpy(s.read_unsorted())
    morton_diff = float(np.abs(runm(True) - runm(False)).max())
    print(f"   morton invariance max|Δ| = {morton_diff:.2e}")

    # (d) conservation, dynamic λ=0
    cpos = np.array([64.,64.]) + np.array([[-3,-3],[3,-3],[-3,3],[3,3]], float)
    cs2 = FieldSystemGPU(ConfigV2(H=128, W=128, lam=0.0, mode="dynamic", sort_units=False),
                         positions=cpos, check_mem=False)
    cs2.clamped[:] = True; cs2.a[:] = 1.0
    integ = []
    for _ in range(200):
        cs2.step(); integ.append(float(cs2.phi.sum()))
    tt = np.arange(1, 201); sl, ic = np.polyfit(tt, integ, 1)
    cons_rel = float(np.abs(np.array(integ) - (sl*tt+ic)).max() / (abs(integ[-1]) + 1e-30))
    print(f"   conservation: slope={sl:.4f} (expect 4.0), linear drift rel={cons_rel:.2e}")

    ok = (d_diff < tol and falloff_diff < tol and abs(L_fit-2.0)/2.0 < 0.05
          and morton_diff < tol and cons_rel < 1e-3
          and np.abs(f_disc - cpu_spec).max() < 0.05)  # discrete≈stage1-ish sanity
    print(f"   [{'PARITY OK' if ok else 'PARITY FAIL — STOP'}] "
          f"(float32 tol {tol:.0e}; GPU sums reorder via atomics)")
    return ok


# ===========================================================================
# STEP 3 — CORRECT GPU TIMING
# ===========================================================================
def time_call(fn, warmup=25, iters=40):
    """Median (and IQR) ms per call, done correctly:
    warmup (JIT/autotune/cuFFT-plan), device-synchronize, CUDA-event timing."""
    for _ in range(warmup):
        fn()
    sync()
    ts = np.empty(iters)
    if GPU:
        ev0, ev1 = xp.cuda.Event(), xp.cuda.Event()
        for i in range(iters):
            ev0.record(); fn(); ev1.record(); ev1.synchronize()
            ts[i] = xp.cuda.get_elapsed_time(ev0, ev1)        # ms, device-side
    else:
        for i in range(iters):
            t0 = time.perf_counter(); fn(); sync()
            ts[i] = (time.perf_counter() - t0) * 1e3
    return float(np.median(ts)), float(np.percentile(ts, 25)), float(np.percentile(ts, 75))


def stage_breakdown(sys):
    """Per-stage median ms (deposit / FFT-solve / read / update), each synced."""
    S = sys.deposit(); sys.evolve(S); pr = sys.read()         # prime state
    d, _, _ = time_call(sys.deposit)
    e, _, _ = time_call(lambda: sys.evolve(S))
    r, _, _ = time_call(sys.read)
    u, _, _ = time_call(lambda: sys.update(pr))
    return dict(deposit=d, fft=e, read=r, update=u)


def device_used_bytes():
    if GPU:
        free, total = xp.cuda.runtime.memGetInfo()
        return total - free
    return 0


# ===========================================================================
# STEP 4 — THE MEASUREMENT (GPU only)
# ===========================================================================
def n_sweep(hw, grid=256, budget_ms=16.0):
    print(f"\n[SWEEP] grid {grid}², steady mode, real-time budget {budget_ms} ms/step")
    print(f"   {'N':>12}  {'median ms':>10} {'IQR ms':>14}  "
          f"{'deposit':>8} {'fft':>7} {'read':>8} {'upd':>6}  {'VRAM GB':>8}")
    bytes_per_unit = estimate_memory(ConfigV2(H=grid, W=grid, N=1))[1]  # unit_bytes at N=1
    vram_cap = 0.80 * hw["vram_gb"] * 1e9
    rows = []
    Ns = [1e4, 1e5, 3e5, 1e6, 3e6, 1e7, 3e7, 1e8, 3e8]
    for N in [int(x) for x in Ns]:
        est = estimate_memory(ConfigV2(H=grid, W=grid, N=N))[0]
        if est > vram_cap:
            print(f"   {N:>12,}  — est {est/1e9:.1f} GB > 80% VRAM ({vram_cap/1e9:.1f} GB); stop (memory).")
            rows.append((N, None, None, None, None, "OOM"))
            break
        cfg = ConfigV2(H=grid, W=grid, N=N, N_CAP=2_000_000_000, mem_limit_bytes=vram_cap)
        sys = FieldSystemGPU(cfg)
        med, q1, q3 = time_call(sys.step)
        bd = stage_breakdown(sys)
        used = device_used_bytes() / 1e9
        rows.append((N, med, (q1, q3), bd, used, "ok"))
        print(f"   {N:>12,}  {med:>10.3f} [{q1:6.3f},{q3:6.3f}]  "
              f"{bd['deposit']:>8.3f} {bd['fft']:>7.3f} {bd['read']:>8.3f} "
              f"{bd['update']:>6.3f}  {used:>8.2f}")
        del sys
        if GPU:
            xp.get_default_memory_pool().free_all_blocks()
        if med > budget_ms:
            print(f"   N={N:,} exceeds {budget_ms} ms — real-time ceiling crossed.")
            break
    return rows


def grid_sweep(hw, N=300_000):
    print(f"\n[GRID SWEEP] N={N:,}: confirm FFT stays negligible vs deposit/read")
    for g in [64, 128, 256, 512]:
        cfg = ConfigV2(H=g, W=g, N=N, N_CAP=2_000_000_000,
                       mem_limit_bytes=0.8 * hw["vram_gb"] * 1e9)
        sys = FieldSystemGPU(cfg)
        bd = stage_breakdown(sys)
        print(f"   {g:>3}²  deposit {bd['deposit']:7.3f}  fft {bd['fft']:7.3f}  "
              f"read {bd['read']:7.3f}  update {bd['update']:6.3f}  ms  "
              f"(FFT {100*bd['fft']/sum(bd.values()):.1f}% of step)")
        del sys
        if GPU:
            xp.get_default_memory_pool().free_all_blocks()


def analyze_headline(rows, hw, budget_ms=16.0):
    ok = [r for r in rows if r[5] == "ok"]
    under = [r for r in ok if r[1] < budget_ms]
    largest_rt = under[-1] if under else None
    oom = next((r for r in rows if r[5] == "OOM"), None)
    bottleneck = None
    if ok:
        last = ok[-1][3]
        bottleneck = max(last, key=last.get)
    bytes_per_unit = estimate_memory(ConfigV2(N=1))[1]
    n_mem = int(0.80 * hw["vram_gb"] * 1e9 / bytes_per_unit)
    binds = "compute (16 ms)" if (largest_rt and largest_rt[0] < n_mem) else "memory"
    print("\n" + "=" * 74)
    print("HEADLINE NUMBERS")
    print("=" * 74)
    print(f"  (1) largest N under {budget_ms} ms/step : "
          f"{largest_rt[0]:,} ({largest_rt[1]:.2f} ms)" if largest_rt else "  (1) none under budget")
    print(f"  (2) memory-out N (80% VRAM)        : ~{n_mem:,}"
          + (f"  (hit OOM at N={oom[0]:,})" if oom else ""))
    print(f"  (3) binding constraint on this card: {binds}")
    print(f"  (4) bottleneck stage at the ceiling: {bottleneck}")
    return dict(largest_rt=largest_rt, n_mem=n_mem, binds=binds, bottleneck=bottleneck)


# ===========================================================================
# STEP 5 — MEMORY CEILINGS (reliable) + BANDWIDTH MODEL (labeled estimate)
# ===========================================================================
# Bandwidth model: the step streams the (N,K)-sized arrays through HBM a small
# number of times (win_ker, win_idx, the shared buffer, the gathered φ) — the
# 256² grid itself is L2-resident, and the FFT is a tiny constant. So
#     bytes/step ≈ C_TRAFFIC · N · K · 4 ,   t_step ≈ bytes/step / (UTIL·BW_peak)
# C_TRAFFIC and UTIL are the two stated assumptions; treat the resulting N as an
# order-of-magnitude ESTIMATE, not a measurement.
C_TRAFFIC = 10.0      # # of (N,K) array passes through HBM per step (deposit+read)
UTIL = 0.6            # fraction of peak bandwidth realized by scatter/gather


def ceilings_table(K=169, budget_ms=16.0):
    bytes_per_unit = estimate_memory(ConfigV2(N=1))[1]                # exact, reliable
    print("\n" + "=" * 74)
    print("CEILINGS — memory is exact; the 16 ms (compute) column is a LABELED MODEL")
    print(f"  bytes/unit = {bytes_per_unit} (exact)   |   model: bytes/step ≈ "
          f"{C_TRAFFIC:.0f}·N·K·4, util {UTIL:.0%} of peak BW")
    print("=" * 74)
    print(f"  {'GPU':<26} {'VRAM':>5} {'BW':>7}  {'N_mem(80%)':>12}  {'N_16ms(model)':>14}  binds")
    out = {}
    for name, (vram, bw) in _GPU_DB.items():
        n_mem = 0.80 * vram * 1e9 / bytes_per_unit
        n_cmp = (budget_ms * 1e-3) * (UTIL * bw * 1e9) / (C_TRAFFIC * K * 4)
        binds = "compute" if n_cmp < n_mem else "memory"
        out[name] = (n_mem, n_cmp, binds)
        print(f"  {name:<26} {vram:>3}GB {bw:>5}GB/s  {n_mem:>12,.0f}  {n_cmp:>14,.0f}  {binds}")
    print("  (N_mem is reliable: fixed bytes/unit. N_16ms is an order-of-magnitude")
    print("   bandwidth model — the real value must be MEASURED on the actual card.)")
    return out


def blackwell_extrapolation(headline, hw, ceilings):
    print("\n" + "=" * 74)
    print("EXTRAPOLATION TO RTX 6000 Pro Blackwell (96 GB) — labeled estimate")
    print("=" * 74)
    bytes_per_unit = estimate_memory(ConfigV2(N=1))[1]
    bw_b, vram_b = 1792, 96
    n_mem_b = int(0.80 * vram_b * 1e9 / bytes_per_unit)
    if headline["binds"].startswith("compute"):
        # bandwidth-bound: scale the MEASURED Colab ceiling by the BW ratio
        n_rt = headline["largest_rt"][0]
        ratio = bw_b / hw["bw"]
        n_b = int(n_rt * ratio)
        print(f"  This card is COMPUTE(bandwidth)-bound at N≈{n_rt:,}.")
        print(f"  Blackwell BW {bw_b} / {hw['name']} BW {hw['bw']:.0f} = ×{ratio:.2f}")
        print(f"  → ESTIMATE real-time N ≈ {n_b:,}  (bandwidth-ratio scaling; an ESTIMATE,")
        print(f"    rests on the workload being bandwidth-bound — verify by measuring).")
        print(f"  (Memory would allow ~{n_mem_b:,}, so compute still binds first.)")
    else:
        n_rt = headline["largest_rt"][0] if headline["largest_rt"] else headline["n_mem"]
        ratio = vram_b / hw["vram_gb"]
        n_b = int(headline["n_mem"] * ratio)
        print(f"  This card is MEMORY-bound at N≈{headline['n_mem']:,}.")
        print(f"  Blackwell VRAM 96 / {hw['vram_gb']:.0f} GB = ×{ratio:.2f}  (linear, reliable)")
        print(f"  → real-time N ≈ {n_b:,}  (memory scaling — reliable since bytes/unit fixed).")


# ===========================================================================
# PLOTS
# ===========================================================================
def plot_measured(rows, hw, budget_ms, path):
    ok = [r for r in rows if r[5] == "ok"]
    Ns = np.array([r[0] for r in ok], float); ts = np.array([r[1] for r in ok])
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.loglog(Ns, ts, "o-", label="median ms/step")
    ax.axhline(budget_ms, color="C3", ls="--", label=f"{budget_ms} ms real-time budget")
    under = Ns[ts < budget_ms]
    if len(under):
        ax.axvline(under[-1], color="C2", ls=":", label=f"ceiling N≈{int(under[-1]):,}")
    ax.set_xlabel("N (units)"); ax.set_ylabel("median time / step (ms)")
    ax.set_title(f"Stage-2b MEASURED on {hw['name']} ({hw['vram_gb']:.0f} GB) — grid 256²")
    ax.grid(True, which="both", ls=":", alpha=0.5); ax.legend()
    fig.tight_layout(); fig.savefig(path, dpi=110); plt.close(fig)


def plot_projection(ceilings, path, K=169, budget_ms=16.0):
    """No GPU here → a clearly-labeled MODEL plot: modeled t_step(N) for a few
    representative cards, the 16 ms line, and each card's memory ceiling."""
    bytes_per_unit = estimate_memory(ConfigV2(N=1))[1]
    fig, ax = plt.subplots(figsize=(7, 5))
    Ns = np.logspace(4, 8, 60)
    for name, color in [("Tesla T4", "C0"), ("A100-SXM4-80GB", "C1"),
                        ("RTX 6000 Pro Blackwell", "C2")]:
        vram, bw = _GPU_DB[name]
        t = (C_TRAFFIC * Ns * K * 4) / (UTIL * bw * 1e9) * 1e3       # ms, MODELED
        ax.loglog(Ns, t, color=color, label=f"{name} (~{bw} GB/s)")
        n_mem = 0.80 * vram * 1e9 / bytes_per_unit
        ax.axvline(n_mem, color=color, ls=":", alpha=0.5)
    ax.axhline(budget_ms, color="k", ls="--", label=f"{budget_ms} ms budget")
    ax.set_xlabel("N (units)"); ax.set_ylabel("MODELED time / step (ms)")
    ax.set_title("Stage-2b PROJECTION (bandwidth model — NOT a GPU measurement)\n"
                 f"assumptions: {C_TRAFFIC:.0f}·N·K·4 bytes/step, util {UTIL:.0%}; "
                 "dotted = memory ceilings")
    ax.grid(True, which="both", ls=":", alpha=0.5); ax.legend(fontsize=8)
    fig.tight_layout(); fig.savefig(path, dpi=110); plt.close(fig)


# ===========================================================================
# main
# ===========================================================================
def main():
    hw = hardware_report()

    if GPU:
        parity = correctness_parity()
        if not parity:
            print("\nPARITY FAILED — refusing to report timings (a port bug must not "
                  "be hidden by moving on). Investigate before timing.")
            raise SystemExit(1)
        rows = n_sweep(hw)
        grid_sweep(hw)
        headline = analyze_headline(rows, hw)
        ceil = ceilings_table()
        blackwell_extrapolation(headline, hw, ceil)
        plot_measured(rows, hw, 16.0, "field_v2b_realtime_ceiling.png")
        print("\n[FIGURE] field_v2b_realtime_ceiling.png")
        rt = headline["largest_rt"]
        print("\n" + "#" * 74)
        print("# 5-LINE SUMMARY")
        print(f"#  GPU                : {hw['name']} ({hw['vram_gb']:.0f} GB, ~{hw['bw']:.0f} GB/s)")
        print(f"#  largest real-time N: {rt[0]:,} under 16 ms/step" if rt else "#  largest real-time N: none")
        print(f"#  binding constraint : {headline['binds']}")
        print(f"#  bottleneck stage   : {headline['bottleneck']}")
        b = 1792 / hw["bw"] if headline["binds"].startswith("compute") else 96/hw["vram_gb"]
        print(f"#  Blackwell 96 GB    : ~×{b:.1f} of measured (labeled estimate; see above)")
        print("#" * 74)
    else:
        # CPU sandbox: validate the PORT, then give reliable memory ceilings + a
        # labeled bandwidth model. No GPU time is ever reported as a measurement.
        parity = correctness_parity()
        ceil = ceilings_table()
        plot_projection(ceil, "field_v2b_realtime_ceiling.png")
        print("\n[FIGURE] field_v2b_realtime_ceiling.png  (PROJECTION — labeled model)")
        bytes_per_unit = estimate_memory(ConfigV2(N=1))[1]
        n_mem_b = int(0.80 * 96 * 1e9 / bytes_per_unit)
        print("\n" + "#" * 74)
        print("# 5-LINE SUMMARY")
        print(f"#  GPU                : NONE in this sandbox — port validated on CPU, "
              f"parity {'OK' if parity else 'FAIL'}")
        print(f"#  largest real-time N: MEASURE ON COLAB (harness ready). Model 16 ms ceiling:")
        print(f"#                       ~0.45M (T4) · ~2.5M (A100/Blackwell) · ~4.8M (H100)")
        print(f"#  binding constraint : compute(bandwidth) binds first on every card, below")
        print(f"#                       the memory ceiling (~6M T4 · ~31M A100-80 · ~37M Blackwell)")
        print(f"#  bottleneck stage   : deposit/read (scatter/gather, bandwidth) — not the FFT")
        print(f"#  Blackwell 96 GB    : memory allows ~{n_mem_b:,}; real-time N stays BW-bound,")
        print(f"#                       est ~2.5M (= measured Colab N × Blackwell/Colab BW ratio)")
        print("#" * 74)
        raise SystemExit(0 if parity else 1)


if __name__ == "__main__":
    main()


## 4. Run the measurement
Run as a subprocess so the harness's clean `SystemExit` doesn't surface as a notebook traceback. This is the cell whose **full output you paste back**.

In [ ]:
!python field_system_v2_gpu.py

## 5. Show the measured ceiling plot

In [ ]:
import os
from IPython.display import Image, display
p = 'field_v2b_realtime_ceiling.png'
display(Image(p)) if os.path.exists(p) else print('no figure — did cell 4 run on a GPU?')
